<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_3/notebooks/Unit_3_Data_Analysis_Assignment_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Unit 3 Data Analysis Assignment — Calcium Imaging of a Drug Wash-On

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**  ·  *Assignment (10 points)*

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## Background

In W10 L1 you showed that **4-AP**, a K⁺ channel blocker, made neurons fire calcium events far more
often than the vehicle did.

The lab has now received a **new, uncharacterised compound**. Nobody knows yet what it does: it might
raise activity like 4-AP, lower it, or leave it unchanged. Your job is to run the same analysis you
learned in class and find out — and to read the sign of every number rather than assume it.

## The data

Two brain slices were imaged, **100 neurons** each, for **6 minutes at 10 frames per second**. Three
minutes in, a solution was washed on:

| Group | What was washed on at 180 s | Why it's there |
|---|---|---|
| **CONTROL** | **aCSF** — the vehicle, no drug | shows what "no drug" looks like over the same six minutes |
| **DRUG** | the **new compound** | the condition you are asking about |

Each recording has a **baseline period (0–180 s)** and an **application period (180–360 s)**. The CSV
has **one row per frame and one column per cell** (`C1`–`C100`).

## How this assignment works

**Run every cell in order, from the top** (click a cell, press **Shift + Enter**).

| | What to do |
|---|---|
| **⚙️ Try it** | A value in the code that **you change and re-run**. They are optional exploration, **except** the one in Step 4, which Discussion question 4 needs. |
| **💬 Discussion** | At the very end. Give numbers exactly as the notebook prints them; the other answers are short. |

**Points (10 total):** Steps 1–4 are worth **5 points** (marked on each step), earned by running its cells
so the outputs show in your PDF. The Discussion is worth **5 points**, 1 point per question.

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Load the data and look at raw brightness [1 pt]

> **⚙️ Try it (optional) — inspect a different cell.** Set `CELL_TO_INSPECT = "C42"` and re-run. Notice
> how its resting brightness differs from C1's, then set it back to `"C1"`.

In [ ]:
import os
import numpy as np                # for working with arrays of numbers
import pandas as pd               # for the data table
import matplotlib.pyplot as plt   # for every figure in this notebook

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_3/data/"

FPS = 10          # frames per second: one frame every 0.1 s
N_CELLS = 100     # cells per recording (the columns C1..C100)

# Load the table, falling back to a hand-uploaded copy if the download fails
FILENAME = "functional_imaging_assignment.csv"
try:
    df = pd.read_csv(DATA + FILENAME)
except Exception:
    if not os.path.exists(FILENAME):
        print("Could not download " + FILENAME + ". Choose that file from your computer when asked:")
        try:
            from google.colab import files      # only exists inside Google Colab
            files.upload()
        except ImportError:
            pass
    if not os.path.exists(FILENAME):
        raise FileNotFoundError(
            "Could not find " + FILENAME + ". Click the folder icon in the left sidebar, "
            "upload it, then run this cell again.")
    df = pd.read_csv(FILENAME)

# The cell columns are the ones named C1, C2, ... C100. We reuse this list constantly.
cell_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]

print("Combined table (rows, columns):", df.shape)
print("Rows per recording:            ", df.groupby("Group").size().to_dict())
print("Cells per recording:           ", len(cell_cols))
print("Recording length:              ", round(df["Time"].max() + 1 / FPS), "s")

df.head()

In [ ]:
CELL_TO_INSPECT = "C1"    # ⚙️ Try it — any cell from "C1" to "C100"

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, colour in [("CONTROL", "gray"), ("DRUG", "tomato")]:
    sub = df[df["Group"] == name]
    ax.plot(sub["Time"], sub[CELL_TO_INSPECT], color=colour, lw=0.6,
            label="%s recording, cell %s" % (name, CELL_TO_INSPECT))
ax.axvline(180, color="black", ls="--", lw=1.2)
ax.text(181, ax.get_ylim()[1], " wash-on", va="top", fontsize=9)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Raw fluorescence (a.u.)")
ax.set_title("Raw brightness of cell %s" % CELL_TO_INSPECT)
ax.legend(fontsize=8, loc="upper left")
plt.show()

# Resting brightness = the average raw value during the baseline (first 180 s)
resting = df[(df["Group"] == "DRUG") & (df["Time"] < 180)][cell_cols].mean()
print("Resting raw brightness in the DRUG recording:")
print("   %-4s %.0f   <- the cell you are inspecting" % (CELL_TO_INSPECT, resting[CELL_TO_INSPECT]))
print("   dimmest cell:   %s  %.0f" % (resting.idxmin(), resting.min()))
print("   brightest cell: %s  %.0f" % (resting.idxmax(), resting.max()))

## Step 2 — Convert brightness into ΔF/F₀ [1 pt]

$$\frac{\Delta F}{F_0} = \frac{F - F_0}{F_0}\times 100\%$$

**F₀** is each cell's own resting brightness, averaged over the baseline window. **0%** means no change
from rest, a **positive** value means the cell is brighter (more active) than its own rest, and a
**negative** value means it is dimmer (less active). The baseline must come out at ≈ 0% — that is a
check, not a result.

> **⚙️ Try it (optional) — move the F₀ window.** Set `F0_WINDOW_SEC = 360` and re-run. Notice what happens
> to the **DRUG baseline** number, then set it back to **180** and re-run before moving on.

In [ ]:
F0_WINDOW_SEC = 180     # ⚙️ Try it — the window (s) that defines F0. Try 360, then set back to 180.


def compute_dff(group_name):
    """Return a Time + ΔF/F₀ table for one recording (CONTROL or DRUG)."""
    sub      = df[df["Group"] == group_name].reset_index(drop=True)
    baseline = sub[sub["Time"] < F0_WINDOW_SEC]      # the frames that define rest
    F0       = baseline[cell_cols].mean()            # each cell's own resting brightness
    dff      = (sub[cell_cols] - F0) / F0 * 100      # (F - F0) / F0, as a percentage
    return pd.concat([sub[["Time"]], dff], axis=1)   # keep Time alongside the signal


dff_con  = compute_dff("CONTROL")
dff_drug = compute_dff("DRUG")

print("F0_WINDOW_SEC = %g s\n" % F0_WINDOW_SEC)
print("Average ΔF/F₀ in each half of each recording:")
for name, d in [("CONTROL", dff_con), ("DRUG", dff_drug)]:
    base = d.loc[d["Time"] <  180, cell_cols].values.mean()
    app  = d.loc[d["Time"] >= 180, cell_cols].values.mean()
    print("   %-8s baseline = %5.2f %%   application = %5.2f %%" % (name, base, app))

## Step 3 — Look at the population and at single cells [1 pt]

The heatmap shows all 100 cells (rows) over time, coloured by ΔF/F₀. The trace plot stacks 12 cells so
you can see the shape of each event. The dashed line marks the wash-on at 180 s.

> **⚙️ Try it (optional) — change the colour scale.** Try `COLOUR_MAX = 20` and `150`. Set it back to **60**.

In [ ]:
COLOUR_MAX = 60      # ⚙️ Try it — the ΔF/F₀ (%) shown as full brightness. Try 20, then 150.

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (name, d) in zip(axes, [("CONTROL", dff_con), ("DRUG", dff_drug)]):
    # .T puts cells on rows and time on columns; vmin/vmax keep both panels on one colour scale
    im = ax.imshow(d[cell_cols].values.T, aspect="auto", cmap="viridis",
                   vmin=-20, vmax=COLOUR_MAX, extent=[0, d["Time"].max(), N_CELLS, 1])
    ax.axvline(180, color="white", linestyle="--", linewidth=1.5)   # wash-on
    ax.set_title("%s — ΔF/F₀ heatmap" % name)
    ax.set_xlabel("Time (s)")
axes[0].set_ylabel("Cell number")
fig.colorbar(im, ax=axes, label="ΔF/F₀ (%)", shrink=0.8)
plt.show()

In [ ]:
N_SHOW = 12          # how many cells to stack
OFFSET = 80          # vertical spacing between stacked traces (arbitrary)

show_cells = cell_cols[:N_SHOW]

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, (name, d) in zip(axes, [("CONTROL", dff_con), ("DRUG", dff_drug)]):
    for i, c in enumerate(show_cells):
        ax.plot(d["Time"], d[c] + i * OFFSET, linewidth=0.6)   # i * OFFSET stacks them
    ax.axvline(180, color="black", linestyle="--", linewidth=1.2)
    ax.set_title("%s — individual traces (%d cells)" % (name, N_SHOW))
    ax.set_xlabel("Time (s)")
    ax.set_yticks([])                                          # the offsets are arbitrary
axes[0].set_ylabel("ΔF/F₀ (stacked, arbitrary offset)")
plt.tight_layout()
plt.show()

## Step 4 — Detect events and measure the event rate [2 pts]

An **event** is an upward crossing of `THRESHOLD`, with events at least 1 s apart. The event rate is
the number of events divided by the window length in minutes. The second cell marks every detected
event with a red dot.

> **⚙️ Try it — raise the threshold (needed for Discussion question 4).** Note the rates at
> `THRESHOLD = 20`. Then set `THRESHOLD = 40`, re-run **both** cells below, and compare the **DRUG
> application** rate with before. Set it back to **20** and re-run both.

In [ ]:
THRESHOLD = 20.0                 # ⚙️ Try it — the ΔF/F₀ (%) a cell must cross to count as an event
MIN_GAP_FRAMES = int(1.0 * FPS)  # events must be at least ~1 s apart


def event_frames(trace):
    """Frame numbers of detected events: upward crossings, kept >= MIN_GAP_FRAMES apart."""
    above  = trace > THRESHOLD
    rising = np.where((~above[:-1]) & (above[1:]))[0] + 1   # below, then above = a rising edge
    kept = []
    for r in rising:
        if not kept or r - kept[-1] >= MIN_GAP_FRAMES:
            kept.append(r)
    return np.array(kept, dtype=int)


def event_rate_per_cell(d, t_start, t_end):
    """Events per minute, for every cell, within one time window."""
    mask    = ((d["Time"] >= t_start) & (d["Time"] < t_end)).values
    minutes = (t_end - t_start) / 60.0
    return np.array([len(event_frames(d[c].values[mask])) / minutes for c in cell_cols])


rows = []
for name, d in [("CONTROL", dff_con), ("DRUG", dff_drug)]:
    base = event_rate_per_cell(d,   0, 180)      # baseline window
    app  = event_rate_per_cell(d, 180, 360)      # application window
    rows.append({"Group": name,
                 "Baseline (events/min)":    round(base.mean(), 2),
                 "Application (events/min)": round(app.mean(), 2)})

print("THRESHOLD = %g %% ΔF/F₀\n" % THRESHOLD)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def event_peaks(trace):
    """Same events as event_frames, nudged to each transient's peak so the dots sit on top."""
    win = int(1.5 * FPS)
    return np.array([r + int(np.argmax(trace[r:r + win])) for r in event_frames(trace)], dtype=int)


fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, (name, d) in zip(axes, [("CONTROL", dff_con), ("DRUG", dff_drug)]):
    t = d["Time"].values
    for i, c in enumerate(show_cells):            # same cells as Step 3
        trace = d[c].values
        ax.plot(t, trace + i * OFFSET, linewidth=0.6, color="0.6")
        ev = event_peaks(trace)
        ax.plot(t[ev], trace[ev] + i * OFFSET, "o", color="crimson", markersize=3, zorder=3)
    ax.axvline(180, color="black", linestyle="--", linewidth=1.2)
    ax.set_title("%s — detected events (THRESHOLD = %g%%)" % (name, THRESHOLD))
    ax.set_xlabel("Time (s)")
    ax.set_yticks([])
axes[0].set_ylabel("ΔF/F₀ (stacked) — dots mark detected events")
plt.tight_layout()
plt.show()

### Your results

Your default-setting numbers in one table.

In [ ]:
if F0_WINDOW_SEC != 180 or THRESHOLD != 20:
    print("*** Set F0_WINDOW_SEC back to 180 (Step 2) and THRESHOLD back to 20 (Step 4), ***")
    print("*** then re-run from Step 2 so this table shows the default numbers.        ***\n")

result_rows = []
for name, d in [("CONTROL", dff_con), ("DRUG", dff_drug)]:
    base_rate = event_rate_per_cell(d,   0, 180)
    app_rate  = event_rate_per_cell(d, 180, 360)
    result_rows.append({
        "Group": name,
        "Application_mean_dFF_pct":   round(d.loc[d["Time"] >= 180, cell_cols].values.mean(), 2),
        "Baseline_events_per_min":    round(base_rate.mean(), 2),
        "Application_events_per_min": round(app_rate.mean(), 2),
    })

my_result = pd.DataFrame(result_rows).set_index("Group")
my_result.astype(object).T      # .T flips the table on its side so it is easier to read

---

### 💬 Discussion [5 pts]

Give numbers exactly as the notebook prints them. Use the default ⚙️ values unless a question tells you
to change one. Each question is worth **1 point**.

1. **Resting raw brightness (Step 1) [1 pt].** What is the resting raw brightness of the **dimmest cell** in the
   **DRUG** recording, to the nearest whole number?

2. **Mean ΔF/F₀ (Step 2) [1 pt].** What are the **CONTROL** and **DRUG** mean ΔF/F₀ during the
   application period, in %?

3. **Event rate (Step 4) [1 pt].** Identify the application-period event rate for **DRUG** and for
   **CONTROL**, in events per minute. Which one is higher?

4. **Threshold (Step 4) [1 pt].** Set `THRESHOLD` to `40` and re-run both Step 4 cells. Does the **DRUG**
   application event rate go **up**, go **down**, or **stay the same** compared with `THRESHOLD = 20`?
   Explain why in one sentence.

5. **True or false (Steps 2–4) [1 pt].** Compared with the CONTROL slice, the new compound **decreased**
   calcium activity after wash-on, meaning the neurons were **less active**.

*(Before you submit: set `THRESHOLD = 20` and put any other ⚙️ values you changed back to their defaults,
then run all.)*

**Your answer:** *(double-click here to type)*

>
>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.